In [2]:
import pandas as pd
import numpy as np

# Load the actual financial anomaly feature dataset
df = pd.read_csv("financial_anomaly_features.csv", low_memory=False)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())


Dataset loaded successfully.
Rows: 98755
Columns: 45


,work_id,house,mp_key,state,ida,work_category,work_title,financial_year,mp_number_from_work_id,work_serial,...,sanction_minus_completion_amount,sanction_minus_completion_percentage,image_indicator,peer_median_sanction_amount,peer_mean_sanction_amount,peer_median_payment_count,peer_median_payment_amount,sanction_amount_vs_peer_median,payment_count_vs_peer_median,disbursed_amount_vs_peer_median
0,WS/MP620/2024-2025/133166,Lok Sabha,LS_PRALHAD_VENKATESH_JOSHI,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Normal/Others,Construction of buildings for community cultur...,2024-2025,MP620,133166,...,0.0,0.0,1.0,300000.0,523203.608126,1.0,199276.5,1.657283,1.0,2.494950
1,WS/MP620/2025-2026/133167,Lok Sabha,LS_PRALHAD_VENKATESH_JOSHI,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Trust and Society,Construction of rooms and halls in school and ...,2025-2026,MP620,133167,...,NaN,NaN,NaN,649825.0,978469.076596,1.0,411500.0,0.769438,0.0,0.000000
2,WS/MP620/2024-2025/133190,Lok Sabha,LS_PRALHAD_VENKATESH_JOSHI,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Trust and Society,Construction of buildings for community cultur...,2024-2025,MP620,133190,...,NaN,NaN,NaN,649825.0,978469.076596,1.0,411500.0,0.692494,0.0,0.000000
3,WS/MP620/2025-2026/133191,Lok Sabha,LS_PRALHAD_VENKATESH_JOSHI,Karnataka,HAVERI(DEPUTY COMMISSIONER HAVERI_IDA),Normal/Others,Construction of buildings for community cultur...,2025-2026,MP620,133191,...,NaN,NaN,NaN,300000.0,523203.608126,1.0,199276.5,5.000000,1.0,4.269992
4,WS/MP620/2024-2025/133301,Lok Sabha,LS_PRALHAD_VENKATESH_JOSHI,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Normal/Others,Construction of buildings for community cultur...,2024-2025,MP620,133301,...,0.0,0.0,1.0,300000.0,523203.608126,1.0,199276.5,1.326697,1.0,1.997270


In [3]:
# Columns that should NOT be used as ML features

remove_columns = [
    # Identifiers / descriptive information
    "work_id",
    "house",
    "mp_key",
    "state",
    "ida",
    "work_category",
    "work_title",
    "mp_number_from_work_id",
    "work_serial",
    "constituency",
    "elected_nominated",

    # Current status / post-outcome information
    "work_status",
    "completion_date",
    "completion_amount",
    "completion_to_sanction_ratio",
    "sanction_minus_completion_amount",
    "sanction_minus_completion_percentage",
    "image_indicator",

    # Raw dates
    "sanction_date",
    "first_payment_date",
    "last_payment_date",

    # Constant feature — contains only one value
    "payment_exceeds_sanction_indicator"
]

# Create a copy
financial_clean = df.copy()

# Remove unwanted columns
financial_clean = financial_clean.drop(columns=remove_columns)

print("Original columns:", df.shape[1])
print("Remaining columns:", financial_clean.shape[1])

print("\nRemaining features:")
for col in financial_clean.columns:
    print(col)

Original columns: 45
Remaining columns: 23

Remaining features:
financial_year
sanction_amount
total_disbursed_amount
payment_count
successful_payment_count
in_progress_payment_count
successful_payment_amount
in_progress_payment_amount
payment_to_sanction_ratio
payment_minus_sanction_amount
average_payment_amount
median_payment_amount
maximum_payment_amount
minimum_payment_amount
payment_amount_std
payment_duration_days
peer_median_sanction_amount
peer_mean_sanction_amount
peer_median_payment_count
peer_median_payment_amount
sanction_amount_vs_peer_median
payment_count_vs_peer_median
disbursed_amount_vs_peer_median


In [4]:
print("Shape:", financial_clean.shape)

print("\nData types:")
display(financial_clean.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing = pd.DataFrame({
    "missing_count": financial_clean.isna().sum(),
    "missing_percentage": financial_clean.isna().mean() * 100
})

display(missing[missing["missing_count"] > 0].sort_values(
    "missing_percentage", ascending=False
))

Shape: (98755, 23)

Data types:


,dtype
financial_year,str
sanction_amount,float64
total_disbursed_amount,float64
payment_count,int64
successful_payment_count,int64
in_progress_payment_count,int64
successful_payment_amount,float64
in_progress_payment_amount,float64
payment_to_sanction_ratio,float64
payment_minus_sanction_amount,float64



Missing values:


,missing_count,missing_percentage
payment_amount_std,79195,80.193408
average_payment_amount,26868,27.206724
median_payment_amount,26868,27.206724
maximum_payment_amount,26868,27.206724
minimum_payment_amount,26868,27.206724
payment_duration_days,26868,27.206724
peer_median_sanction_amount,7,0.007088
peer_mean_sanction_amount,7,0.007088
peer_median_payment_count,7,0.007088
peer_median_payment_amount,7,0.007088


In [5]:
# Create a dataset for modelling while keeping work_id only for identification
model_data = pd.concat(
    [
        df[["work_id", "financial_year"]],
        financial_clean.drop(columns=["financial_year"], errors="ignore")
    ],
    axis=1
)

print("Model dataset shape:", model_data.shape)

display(model_data.head())

Model dataset shape: (98755, 24)


,work_id,financial_year,sanction_amount,total_disbursed_amount,payment_count,successful_payment_count,in_progress_payment_count,successful_payment_amount,in_progress_payment_amount,payment_to_sanction_ratio,...,minimum_payment_amount,payment_amount_std,payment_duration_days,peer_median_sanction_amount,peer_mean_sanction_amount,peer_median_payment_count,peer_median_payment_amount,sanction_amount_vs_peer_median,payment_count_vs_peer_median,disbursed_amount_vs_peer_median
0,WS/MP620/2024-2025/133166,2024-2025,497185.0,497185.0,1,1,0,497185.0,0.0,1.000000,...,497185.0,NaN,0.0,300000.0,523203.608126,1.0,199276.5,1.657283,1.0,2.494950
1,WS/MP620/2025-2026/133167,2025-2026,500000.0,0.0,0,0,0,0.0,0.0,0.000000,...,NaN,NaN,NaN,649825.0,978469.076596,1.0,411500.0,0.769438,0.0,0.000000
2,WS/MP620/2024-2025/133190,2024-2025,450000.0,0.0,0,0,0,0.0,0.0,0.000000,...,NaN,NaN,NaN,649825.0,978469.076596,1.0,411500.0,0.692494,0.0,0.000000
3,WS/MP620/2025-2026/133191,2025-2026,1500000.0,850909.0,1,1,0,850909.0,0.0,0.567273,...,850909.0,NaN,0.0,300000.0,523203.608126,1.0,199276.5,5.000000,1.0,4.269992
4,WS/MP620/2024-2025/133301,2024-2025,398009.0,398009.0,1,1,0,398009.0,0.0,1.000000,...,398009.0,NaN,0.0,300000.0,523203.608126,1.0,199276.5,1.326697,1.0,1.997270


In [6]:
# Chronological train/test split
# Older financial years -> training
# Latest financial year -> testing

train_df = model_data[
    model_data["financial_year"] != "2026-2027"
].copy()

test_df = model_data[
    model_data["financial_year"] == "2026-2027"
].copy()

print("Training set:", train_df.shape)
print("Testing set :", test_df.shape)

print("\nTraining financial years:")
print(train_df["financial_year"].value_counts().sort_index())

print("\nTesting financial years:")
print(test_df["financial_year"].value_counts().sort_index())

Training set: (78739, 24)
Testing set : (20016, 24)

Training financial years:
financial_year
2023-2024     2953
2024-2025    17921
2025-2026    57865
Name: count, dtype: int64

Testing financial years:
financial_year
2026-2027    20016
Name: count, dtype: int64


In [7]:
train_df.to_csv(
    "financial_anomaly_train.csv",
    index=False
)

test_df.to_csv(
    "financial_anomaly_test.csv",
    index=False
)

print("Files created successfully:")
print("1. financial_anomaly_train.csv")
print("2. financial_anomaly_test.csv")

Files created successfully:
1. financial_anomaly_train.csv
2. financial_anomaly_test.csv


In [8]:
# Investigate why the payment-related features are missing

payment_features = [
    "payment_count",
    "successful_payment_count",
    "in_progress_payment_count",
    "successful_payment_amount",
    "in_progress_payment_amount",
    "average_payment_amount",
    "median_payment_amount",
    "maximum_payment_amount",
    "minimum_payment_amount",
    "payment_amount_std",
    "payment_duration_days"
]

missing_analysis = pd.DataFrame({
    "missing_count": train_df[payment_features].isna().sum(),
    "missing_percentage": train_df[payment_features].isna().mean() * 100
})

display(missing_analysis.sort_values("missing_percentage", ascending=False))

,missing_count,missing_percentage
payment_amount_std,59844,76.002997
median_payment_amount,13824,17.556738
maximum_payment_amount,13824,17.556738
minimum_payment_amount,13824,17.556738
payment_duration_days,13824,17.556738
average_payment_amount,13824,17.556738
payment_count,0,0.000000
in_progress_payment_count,0,0.000000
successful_payment_count,0,0.000000
successful_payment_amount,0,0.000000


In [9]:
# Check actual payment_count distribution

print("Payment count distribution:")
display(
    train_df["payment_count"]
    .value_counts(dropna=False)
    .sort_index()
)

Payment count distribution:


payment_count
0      13824
1      46020
2      15295
3       1445
4        489
5        290
6        257
7        160
8        138
9        113
10       111
11       116
12        68
13        58
14        49
15        46
16        38
17        23
18        16
19        28
20        20
21        27
22        16
23        12
24         6
25         8
26         9
27         8
28         3
29         7
30         5
31         1
32         2
33         6
34         3
35         5
36         2
37         2
38         1
41         2
42         1
43         1
46         1
47         1
49         1
52         1
57         1
61         1
68         1
191        1
Name: count, dtype: int64

In [10]:
# Investigate why payment_amount_std is missing

std_check = train_df.groupby("payment_count").agg(
    total_works=("work_id", "count"),
    missing_std=("payment_amount_std", "count")
)

std_check["missing_std_percentage"] = (
    std_check["missing_std"] / std_check["total_works"] * 100
)

display(std_check.head(15))

,total_works,missing_std,missing_std_percentage
payment_count,,,
0,13824,0,0.0
1,46020,0,0.0
2,15295,15295,100.0
3,1445,1445,100.0
4,489,489,100.0
5,290,290,100.0
6,257,257,100.0
7,160,160,100.0
8,138,138,100.0


In [11]:
# Check whether the other payment statistics are missing exactly when
# payment_count is zero

features_to_check = [
    "average_payment_amount",
    "median_payment_amount",
    "maximum_payment_amount",
    "minimum_payment_amount",
    "payment_duration_days",
    "payment_amount_std"
]

for feature in features_to_check:
    print(f"\n{feature}")
    
    result = pd.crosstab(
        train_df["payment_count"],
        train_df[feature].isna()
    )
    
    display(result.head(10))


average_payment_amount


average_payment_amount,False,True
payment_count,,
0,0,13824
1,46020,0
2,15295,0
3,1445,0
4,489,0
5,290,0
6,257,0
7,160,0
8,138,0



median_payment_amount


median_payment_amount,False,True
payment_count,,
0,0,13824
1,46020,0
2,15295,0
3,1445,0
4,489,0
5,290,0
6,257,0
7,160,0
8,138,0



maximum_payment_amount


...

minimum_payment_amount,False,True
payment_count,,
0,0,13824
1,46020,0
2,15295,0
3,1445,0
4,489,0
5,290,0
6,257,0
7,160,0
8,138,0



payment_duration_days


payment_duration_days,False,True
payment_count,,
0,0,13824
1,46020,0
2,15295,0
3,1445,0
4,489,0
5,290,0
6,257,0
7,160,0
8,138,0



payment_amount_std


payment_amount_std,False,True
payment_count,,
0,0,13824
1,0,46020
2,15295,0
3,1445,0
4,489,0
5,290,0
6,257,0
7,160,0
8,138,0


In [12]:
# Handle structurally missing payment statistics

train_model = train_df.copy()
test_model = test_df.copy()

# These features are undefined when there are no payments.
# Since payment_count = 0 represents "no payment", use 0 for these fields.

zero_when_no_payment = [
    "average_payment_amount",
    "median_payment_amount",
    "maximum_payment_amount",
    "minimum_payment_amount",
    "payment_duration_days"
]

for col in zero_when_no_payment:
    train_model.loc[
        train_model["payment_count"] == 0, col
    ] = 0

    test_model.loc[
        test_model["payment_count"] == 0, col
    ] = 0


# Standard deviation is undefined for 0 or 1 payment.
# Represent "no measurable variation" as 0.

train_model.loc[
    train_model["payment_count"] <= 1, "payment_amount_std"
] = 0

test_model.loc[
    test_model["payment_count"] <= 1, "payment_amount_std"
] = 0


print("Structural missing-value handling completed.")

Structural missing-value handling completed.


In [13]:
# Check remaining missing values after structural handling

remaining_missing = pd.DataFrame({
    "train_missing": train_model.isna().sum(),
    "train_missing_%": train_model.isna().mean() * 100,
    "test_missing": test_model.isna().sum(),
    "test_missing_%": test_model.isna().mean() * 100
})

display(
    remaining_missing[
        (remaining_missing["train_missing"] > 0) |
        (remaining_missing["test_missing"] > 0)
    ].sort_values("train_missing", ascending=False)
)

,train_missing,train_missing_%,test_missing,test_missing_%
peer_median_sanction_amount,7,0.00889,0,0.0
peer_mean_sanction_amount,7,0.00889,0,0.0
peer_median_payment_count,7,0.00889,0,0.0
peer_median_payment_amount,7,0.00889,0,0.0
sanction_amount_vs_peer_median,7,0.00889,0,0.0
payment_count_vs_peer_median,7,0.00889,0,0.0
disbursed_amount_vs_peer_median,7,0.00889,0,0.0


In [14]:
from sklearn.impute import SimpleImputer

# Columns that are identifiers/metadata, NOT ML features
metadata_columns = [
    "work_id",
    "financial_year"
]

# Select only the actual ML features
feature_columns = [
    col for col in train_model.columns
    if col not in metadata_columns
]

print("Number of ML features:", len(feature_columns))
print("\nML features:")
for col in feature_columns:
    print(col)

Number of ML features: 22

ML features:
sanction_amount
total_disbursed_amount
payment_count
successful_payment_count
in_progress_payment_count
successful_payment_amount
in_progress_payment_amount
payment_to_sanction_ratio
payment_minus_sanction_amount
average_payment_amount
median_payment_amount
maximum_payment_amount
minimum_payment_amount
payment_amount_std
payment_duration_days
peer_median_sanction_amount
peer_mean_sanction_amount
peer_median_payment_count
peer_median_payment_amount
sanction_amount_vs_peer_median
payment_count_vs_peer_median
disbursed_amount_vs_peer_median


In [15]:
# Create imputer
imputer = SimpleImputer(strategy="median")

# Fit ONLY on training data
X_train = imputer.fit_transform(train_model[feature_columns])

# Apply the same fitted imputer to test data
X_test = imputer.transform(test_model[feature_columns])

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("\nMissing values in X_train:", np.isnan(X_train).sum())
print("Missing values in X_test :", np.isnan(X_test).sum())

X_train shape: (78739, 22)
X_test shape : (20016, 22)

Missing values in X_train: 0
Missing values in X_test : 0


In [16]:
# Analyze the distribution of all 22 ML features

feature_stats = pd.DataFrame({
    "dtype": train_model[feature_columns].dtypes,
    "unique_values": train_model[feature_columns].nunique(),
    "mean": train_model[feature_columns].mean(),
    "median": train_model[feature_columns].median(),
    "std": train_model[feature_columns].std(),
    "min": train_model[feature_columns].min(),
    "max": train_model[feature_columns].max(),
    "skewness": train_model[feature_columns].skew()
})

display(feature_stats)

,dtype,unique_values,mean,median,std,min,max,skewness
sanction_amount,float64,16776,593017.332559,349965.000000,1.184295e+06,2.460000e+00,7.350000e+07,20.935248
total_disbursed_amount,float64,24257,457932.213479,249315.000000,1.046443e+06,0.000000e+00,7.123865e+07,24.205782
payment_count,int64,50,1.277677,1.000000,1.937807e+00,0.000000e+00,1.910000e+02,20.167247
successful_payment_count,int64,50,1.250054,1.000000,1.923892e+00,0.000000e+00,1.910000e+02,20.465005
in_progress_payment_count,int64,20,0.027623,0.000000,2.925085e-01,0.000000e+00,2.300000e+01,38.082167
successful_payment_amount,float64,23791,450167.189069,244036.000000,1.042111e+06,0.000000e+00,7.123865e+07,24.371437
in_progress_payment_amount,float64,858,7765.024410,0.000000,9.167265e+04,0.000000e+00,9.707139e+06,31.888573
payment_to_sanction_ratio,float64,10936,0.784718,1.000000,3.806653e-01,0.000000e+00,1.000000e+00,-1.442597
payment_minus_sanction_amount,float64,10625,-135085.119080,0.000000,5.300292e+05,-4.167875e+07,0.000000e+00,-20.575606
average_payment_amount,float64,24879,350687.268077,200000.000000,5.803096e+05,0.000000e+00,3.256225e+07,10.950687


In [17]:
# Percentage of zero values in each feature

zero_stats = pd.DataFrame({
    "zero_count": (train_model[feature_columns] == 0).sum(),
    "zero_percentage": (train_model[feature_columns] == 0).mean() * 100
})

display(
    zero_stats.sort_values("zero_percentage", ascending=False)
)

,zero_count,zero_percentage
in_progress_payment_count,76968,97.750797
in_progress_payment_amount,76968,97.750797
payment_duration_days,61772,78.451593
payment_amount_std,61441,78.031217
payment_minus_sanction_amount,43762,55.578557
successful_payment_count,15081,19.153152
successful_payment_amount,15081,19.153152
minimum_payment_amount,13824,17.556738
total_disbursed_amount,13824,17.556738
payment_count,13824,17.556738


In [18]:
# Correlation between financial features

correlation_matrix = train_model[feature_columns].corr()

display(correlation_matrix.round(2))

,sanction_amount,total_disbursed_amount,payment_count,successful_payment_count,in_progress_payment_count,successful_payment_amount,in_progress_payment_amount,payment_to_sanction_ratio,payment_minus_sanction_amount,average_payment_amount,...,minimum_payment_amount,payment_amount_std,payment_duration_days,peer_median_sanction_amount,peer_mean_sanction_amount,peer_median_payment_count,peer_median_payment_amount,sanction_amount_vs_peer_median,payment_count_vs_peer_median,disbursed_amount_vs_peer_median
sanction_amount,1.00,0.89,0.13,0.13,0.00,0.89,0.08,-0.02,-0.47,0.63,...,0.54,0.49,0.09,0.12,0.12,0.00,0.12,0.96,0.13,0.87
total_disbursed_amount,0.89,1.00,0.21,0.21,0.01,1.00,0.09,0.20,-0.02,0.71,...,0.61,0.56,0.14,0.11,0.11,0.00,0.11,0.85,0.21,0.98
payment_count,0.13,0.21,1.00,0.99,0.12,0.21,0.01,0.30,0.11,0.01,...,-0.03,0.18,0.30,0.04,0.04,0.00,0.04,0.12,1.00,0.20
successful_payment_count,0.13,0.21,0.99,1.00,-0.03,0.21,-0.04,0.30,0.11,0.01,...,-0.03,0.18,0.30,0.04,0.04,0.00,0.04,0.12,0.99,0.20
in_progress_payment_count,0.00,0.01,0.12,-0.03,1.00,-0.02,0.35,0.03,0.01,-0.01,...,-0.01,0.01,0.03,-0.01,-0.01,0.00,-0.01,0.00,0.12,0.01
successful_payment_amount,0.89,1.00,0.21,0.21,-0.02,1.00,0.00,0.20,-0.02,0.70,...,0.60,0.56,0.14,0.11,0.11,0.00,0.11,0.85,0.21,0.97
in_progress_payment_amount,0.08,0.09,0.01,-0.04,0.35,0.00,1.00,0.03,-0.01,0.11,...,0.11,0.03,0.01,-0.00,-0.00,0.00,-0.00,0.10,0.01,0.11
payment_to_sanction_ratio,-0.02,0.20,0.30,0.30,0.03,0.20,0.03,1.00,0.43,0.27,...,0.25,0.12,0.21,0.04,0.05,0.01,0.04,-0.02,0.30,0.22
payment_minus_sanction_amount,-0.47,-0.02,0.11,0.11,0.01,-0.02,-0.01,0.43,1.00,-0.01,...,-0.01,0.00,0.08,-0.05,-0.04,-0.00,-0.05,-0.46,0.11,-0.02
average_payment_amount,0.63,0.71,0.01,0.01,-0.01,0.70,0.11,0.27,-0.01,1.00,...,0.98,0.24,-0.02,0.11,0.11,0.00,0.11,0.65,0.01,0.74


In [19]:
# Find highly correlated feature pairs

corr_pairs = (
    correlation_matrix
    .where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)

corr_pairs.columns = ["feature_1", "feature_2", "correlation"]

high_corr_pairs = corr_pairs[
    corr_pairs["correlation"].abs() >= 0.90
].sort_values(
    "correlation",
    key=lambda x: x.abs(),
    ascending=False
)

display(high_corr_pairs)

,feature_1,feature_2,correlation
64,payment_count,payment_count_vs_peer_median,1.000000
208,average_payment_amount,median_payment_amount,0.997475
27,total_disbursed_amount,successful_payment_amount,0.996155
370,peer_mean_sanction_amount,peer_median_payment_amount,0.989917
47,payment_count,successful_payment_count,0.988551
86,successful_payment_count,payment_count_vs_peer_median,0.988551
346,peer_median_sanction_amount,peer_mean_sanction_amount,0.980545
232,median_payment_amount,minimum_payment_amount,0.979388
210,average_payment_amount,minimum_payment_amount,0.977325
43,total_disbursed_amount,disbursed_amount_vs_peer_median,0.975335


In [20]:
redundant_features = [
    "payment_count_vs_peer_median",
    "median_payment_amount",
    "successful_payment_amount"
]

In [21]:
# Check peer feature values across financial years

peer_features = [
    "peer_median_sanction_amount",
    "peer_mean_sanction_amount",
    "peer_median_payment_count",
    "peer_median_payment_amount",
    "sanction_amount_vs_peer_median",
    "payment_count_vs_peer_median",
    "disbursed_amount_vs_peer_median"
]

for col in peer_features:
    print(f"\n===== {col} =====")
    
    display(
        train_model.groupby("financial_year")[col]
        .agg(["count", "nunique", "min", "max", "mean"])
    )


===== peer_median_sanction_amount =====


,count,nunique,min,max,mean
financial_year,,,,,
2023-2024,2949,2,500000.0,1000000.0,501017.293998
2024-2025,17919,5,300000.0,1000000.0,350540.081701
2025-2026,57864,7,50000.0,1000000.0,337515.401657



===== peer_mean_sanction_amount =====


,count,nunique,min,max,mean
financial_year,,,,,
2023-2024,2949,3,868153.989973,1.383834e+06,869235.697591
2024-2025,17919,6,523203.608126,1.383834e+06,607790.328244
2025-2026,57864,8,50000.000000,1.383834e+06,583120.406327



===== peer_median_payment_count =====


...

,count,nunique,min,max,mean
financial_year,,,,,
2023-2024,2949,764,0.020000,79.911308,1.300555
2024-2025,17919,5842,0.027440,154.901333,1.726852
2025-2026,57864,11800,0.000008,158.240160,1.737408



===== payment_count_vs_peer_median =====


,count,nunique,min,max,mean
financial_year,,,,,
2023-2024,2949,28,0.0,191.0,1.756189
2024-2025,17919,34,0.0,46.0,1.482002
2025-2026,57864,43,0.0,61.0,1.190015



===== disbursed_amount_vs_peer_median =====


,count,nunique,min,max,mean
financial_year,,,,,
2023-2024,2949,937,0.0,133.185513,2.061337
2024-2025,17919,7274,0.0,237.462170,2.463863
2025-2026,57864,17960,0.0,201.202851,1.916711


In [22]:
# Check whether some features are mathematically derived from others

checks = pd.DataFrame({
    "payment_count": train_model["payment_count"],
    "successful_payment_count": train_model["successful_payment_count"],
    "in_progress_payment_count": train_model["in_progress_payment_count"],
})

checks["count_difference"] = (
    checks["payment_count"]
    - checks["successful_payment_count"]
    - checks["in_progress_payment_count"]
)

print("Payment count relationship:")
display(checks["count_difference"].value_counts().sort_index())

Payment count relationship:


count_difference
0    78739
Name: count, dtype: int64

In [23]:
# Check whether total disbursed amount is exactly
# successful payment amount + in-progress payment amount

amount_check = pd.DataFrame({
    "total_disbursed_amount": train_model["total_disbursed_amount"],
    "successful_payment_amount": train_model["successful_payment_amount"],
    "in_progress_payment_amount": train_model["in_progress_payment_amount"]
})

amount_check["difference"] = (
    amount_check["total_disbursed_amount"]
    - amount_check["successful_payment_amount"]
    - amount_check["in_progress_payment_amount"]
)

print("Difference distribution:")
display(
    amount_check["difference"]
    .value_counts(dropna=False).head(20)
)

print(
    "\nMaximum absolute difference:",
    amount_check["difference"].abs().max()
)

Difference distribution:


difference
0.0    78739
Name: count, dtype: int64


Maximum absolute difference: 0.0


In [24]:
# Check peer features in the test period

for col in peer_features:
    print(f"\n===== {col} =====")
    
    display(
        test_model.groupby("financial_year")[col]
        .agg(["count", "nunique", "min", "max", "mean"])
    )


===== peer_median_sanction_amount =====


,count,nunique,min,max,mean
financial_year,,,,,
2026-2027,20016,6,75000.0,1000000.0,336346.376474



===== peer_mean_sanction_amount =====


,count,nunique,min,max,mean
financial_year,,,,,
2026-2027,20016,7,75000.0,1.383834e+06,582194.021885



===== peer_median_payment_count =====


...

,count,nunique,min,max,mean
financial_year,,,,,
2026-2027,20016,2529,0.000012,165.8,1.800193



===== payment_count_vs_peer_median =====


,count,nunique,min,max,mean
financial_year,,,,,
2026-2027,20016,22,0.0,21.0,0.431255



===== disbursed_amount_vs_peer_median =====


,count,nunique,min,max,mean
financial_year,,,,,
2026-2027,20016,2879,0.0,177.248481,0.934675


In [25]:
# Compare peer feature values between training and testing

peer_comparison = pd.DataFrame({
    "train_unique": train_model[peer_features].nunique(),
    "test_unique": test_model[peer_features].nunique(),
    "train_min": train_model[peer_features].min(),
    "test_min": test_model[peer_features].min(),
    "train_max": train_model[peer_features].max(),
    "test_max": test_model[peer_features].max()
})

display(peer_comparison)

,train_unique,test_unique,train_min,test_min,train_max,test_max
peer_median_sanction_amount,7,6,50000.000000,75000.000000,1.000000e+06,1.000000e+06
peer_mean_sanction_amount,8,7,50000.000000,75000.000000,1.383834e+06,1.383834e+06
peer_median_payment_count,2,2,0.500000,0.500000,1.000000e+00,1.000000e+00
peer_median_payment_amount,8,7,25000.000000,25000.000000,5.000000e+05,5.000000e+05
sanction_amount_vs_peer_median,17370,2529,0.000008,0.000012,1.582402e+02,1.658000e+02
payment_count_vs_peer_median,50,22,0.000000,0.000000,1.910000e+02,2.100000e+01
disbursed_amount_vs_peer_median,25161,2879,0.000000,0.000000,2.374622e+02,1.772485e+02


In [26]:
# ============================================================
# FINAL FEATURE SELECTION FOR FINANCIAL ANOMALY MODEL
# ============================================================

final_features = [
    "sanction_amount",
    "total_disbursed_amount",
    "payment_count",
    "in_progress_payment_count",
    "in_progress_payment_amount",
    "payment_to_sanction_ratio",
    "payment_minus_sanction_amount",
    "average_payment_amount",
    "maximum_payment_amount",
    "minimum_payment_amount",
    "payment_amount_std",
    "payment_duration_days"
]

print("Number of final ML features:", len(final_features))

print("\nFinal ML features:")
for i, feature in enumerate(final_features, 1):
    print(f"{i}. {feature}")

Number of final ML features: 12

Final ML features:
1. sanction_amount
2. total_disbursed_amount
3. payment_count
4. in_progress_payment_count
5. in_progress_payment_amount
6. payment_to_sanction_ratio
7. payment_minus_sanction_amount
8. average_payment_amount
9. maximum_payment_amount
10. minimum_payment_amount
11. payment_amount_std
12. payment_duration_days


In [27]:
# Create final training dataset
financial_anomaly_train_final = train_model[
    ["work_id", "financial_year"] + final_features
].copy()

# Create final testing dataset
financial_anomaly_test_final = test_model[
    ["work_id", "financial_year"] + final_features
].copy()

print("Final training shape:", financial_anomaly_train_final.shape)
print("Final testing shape :", financial_anomaly_test_final.shape)

display(financial_anomaly_train_final.head())

Final training shape: (78739, 14)
Final testing shape : (20016, 14)


,work_id,financial_year,sanction_amount,total_disbursed_amount,payment_count,in_progress_payment_count,in_progress_payment_amount,payment_to_sanction_ratio,payment_minus_sanction_amount,average_payment_amount,maximum_payment_amount,minimum_payment_amount,payment_amount_std,payment_duration_days
0,WS/MP620/2024-2025/133166,2024-2025,497185.0,497185.0,1,0,0.0,1.000000,0.0,497185.0,497185.0,497185.0,0.0,0.0
1,WS/MP620/2025-2026/133167,2025-2026,500000.0,0.0,0,0,0.0,0.000000,-500000.0,0.0,0.0,0.0,0.0,0.0
2,WS/MP620/2024-2025/133190,2024-2025,450000.0,0.0,0,0,0.0,0.000000,-450000.0,0.0,0.0,0.0,0.0,0.0
3,WS/MP620/2025-2026/133191,2025-2026,1500000.0,850909.0,1,0,0.0,0.567273,-649091.0,850909.0,850909.0,850909.0,0.0,0.0
4,WS/MP620/2024-2025/133301,2024-2025,398009.0,398009.0,1,0,0.0,1.000000,0.0,398009.0,398009.0,398009.0,0.0,0.0


In [28]:
# Check missing values in the final feature set

final_missing = pd.DataFrame({
    "train_missing": financial_anomaly_train_final[final_features].isna().sum(),
    "test_missing": financial_anomaly_test_final[final_features].isna().sum()
})

display(
    final_missing[
        (final_missing["train_missing"] > 0) |
        (final_missing["test_missing"] > 0)
    ]
)

,train_missing,test_missing


In [29]:
# Save final datasets used for the Isolation Forest model

financial_anomaly_train_final.to_csv(
    "financial_anomaly_train_final.csv",
    index=False
)

financial_anomaly_test_final.to_csv(
    "financial_anomaly_test_final.csv",
    index=False
)

print("Final datasets saved successfully:")
print("financial_anomaly_train_final.csv")
print("financial_anomaly_test_final.csv")

Final datasets saved successfully:
financial_anomaly_train_final.csv
financial_anomaly_test_final.csv


In [30]:
# ============================================================
# PREPARE FINAL INPUT MATRICES FOR ISOLATION FOREST
# ============================================================

X_train = financial_anomaly_train_final[final_features].copy()
X_test = financial_anomaly_test_final[final_features].copy()

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("\nTraining features:")
print(list(X_train.columns))

print("\nMissing values:")
print("X_train:", X_train.isna().sum().sum())
print("X_test :", X_test.isna().sum().sum())

X_train shape: (78739, 12)
X_test shape : (20016, 12)

Training features:
['sanction_amount', 'total_disbursed_amount', 'payment_count', 'in_progress_payment_count', 'in_progress_payment_amount', 'payment_to_sanction_ratio', 'payment_minus_sanction_amount', 'average_payment_amount', 'maximum_payment_amount', 'minimum_payment_amount', 'payment_amount_std', 'payment_duration_days']

Missing values:
X_train: 0
X_test : 0
